In [ ]:
import os
import subprocess
import sys
import time
from pathlib import Path


In [ ]:
!apt-get install -y ffmpeg

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 3 not upgraded.


In [ ]:
DRIVE_ROOT = Path("/content/drive/MyDrive")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:

SOURCES = {
    "fall": [
        {
            "path": DRIVE_ROOT / "Aegis_Safe_Work/raw/Falls",
            "extensions": {".mp4", ".avi"},
        },
    ],
    "normal": [
        {
            "path": DRIVE_ROOT / "Aegis_Safe_Work/raw/Normal",
            "extensions": {".mp4", ".avi"},
        },
        {
            "path": DRIVE_ROOT / "REAL_LIFE_VIOLENCE/noFights",
            "extensions": {".mp4"},           # .avi descartados de esta fuente
        },
    ],
}

In [ ]:
OUTPUT_ROOT = DRIVE_ROOT / "Aegis_Safe_Work/processed/stage1"

In [ ]:
TARGET_FPS        = 30
TARGET_SIZE       = 224          # letterbox target: 224x224
PAD_COLOR         = "black"      # padding color for letterbox
FFMPEG_CRF        = 23           # H.264 quality (18=high, 23=default, 28=low)
FFMPEG_PRESET     = "fast"       # encoding speed vs compression tradeoff
PREFIX            = {"fall": "Fall", "normal": "Normal"}
ZERO_PAD          = 4            # Fall_0001, Normal_0001


In [ ]:
def check_ffmpeg() -> None:
    result = subprocess.run(
        ["ffmpeg", "-version"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )
    if result.returncode != 0:
        print("[ERROR] ffmpeg not found. Install with: !apt-get install -y ffmpeg")
        sys.exit(1)
    print("[OK] ffmpeg found.")

In [ ]:

def collect_videos(sources: list[dict]) -> list[Path]:
    """Collect all video files from a list of source dicts."""
    videos = []
    for src in sources:
        src_path = Path(src["path"])
        exts     = src["extensions"]
        if not src_path.exists():
            print(f"[WARN] Source path does not exist, skipping: {src_path}")
            continue
        found = sorted(
            p for p in src_path.iterdir()
            if p.is_file() and p.suffix.lower() in exts
        )
        print(f"  {src_path.name}: {len(found)} videos {sorted(exts)}")
        videos.extend(found)
    return videos

In [ ]:

def build_ffmpeg_cmd(src: Path, dst: Path) -> list[str]:
    """
    Build ffmpeg command:
      - Strip audio (-an)
      - Letterbox to TARGET_SIZE x TARGET_SIZE with black padding
      - Force 30 FPS
      - H.264 libx264
      - Overwrite output (-y)
    """
    vf = (
        f"fps={TARGET_FPS},"
        f"scale={TARGET_SIZE}:{TARGET_SIZE}:force_original_aspect_ratio=decrease,"
        f"pad={TARGET_SIZE}:{TARGET_SIZE}:(ow-iw)/2:(oh-ih)/2:color={PAD_COLOR}"
    )
    return [
        "ffmpeg",
        "-y",                        # overwrite output without asking
        "-i", str(src),              # input file
        "-vf", vf,                   # video filter: fps + letterbox
        "-an",                       # strip audio (muted output)
        "-c:v", "libx264",           # H.264 codec
        "-crf", str(FFMPEG_CRF),     # quality
        "-preset", FFMPEG_PRESET,    # encoding speed
        "-movflags", "+faststart",   # web-friendly mp4
        str(dst),                    # output file
    ]


In [ ]:

def process_class(label: str, sources: list[dict], out_dir: Path) -> dict:
    """
    Process all videos for one class label.
    Returns a summary dict.
    """
    out_dir.mkdir(parents=True, exist_ok=True)
    prefix = PREFIX[label]

    print(f"\n[{label.upper()}] Collecting sources...")
    videos = collect_videos(sources)
    total  = len(videos)
    print(f"[{label.upper()}] Total videos to process: {total}")

    ok_count      = 0
    skip_count    = 0
    error_count   = 0
    error_files   = []
    t_start       = time.time()

    for idx, src in enumerate(videos, start=1):
        dst_name = f"{prefix}_{str(idx).zfill(ZERO_PAD)}.mp4"
        dst      = out_dir / dst_name

        # Skip if already processed (resume support)
        if dst.exists() and dst.stat().st_size > 0:
            skip_count += 1
            if idx % 50 == 0:
                print(f"  [{idx}/{total}] SKIP (already exists): {dst_name}")
            continue

        cmd = build_ffmpeg_cmd(src, dst)

        try:
            result = subprocess.run(
                cmd,
                stdout=subprocess.DEVNULL,
                stderr=subprocess.PIPE,
                timeout=120,           # 2 min max per video
            )
            if result.returncode != 0:
                err_msg = result.stderr.decode("utf-8", errors="replace")[-300:]
                print(f"  [{idx}/{total}] ERROR: {src.name}")
                print(f"    ffmpeg stderr: {err_msg}")
                error_count += 1
                error_files.append(str(src))
                # Remove partial output if exists
                if dst.exists():
                    dst.unlink()
            else:
                ok_count += 1
                if idx % 50 == 0 or idx == total:
                    elapsed = time.time() - t_start
                    rate    = ok_count / elapsed if elapsed > 0 else 0
                    eta_s   = (total - idx) / rate if rate > 0 else 0
                    print(
                        f"  [{idx}/{total}] OK: {dst_name} | "
                        f"rate={rate:.1f} vid/s | ETA={eta_s/60:.1f} min"
                    )

        except subprocess.TimeoutExpired:
            print(f"  [{idx}/{total}] TIMEOUT: {src.name}")
            error_count += 1
            error_files.append(str(src))
            if dst.exists():
                dst.unlink()

        except Exception as e:
            print(f"  [{idx}/{total}] EXCEPTION: {src.name} -> {e}")
            error_count += 1
            error_files.append(str(src))

    elapsed_total = time.time() - t_start
    return {
        "label":       label,
        "total_input": total,
        "ok":          ok_count,
        "skipped":     skip_count,
        "errors":      error_count,
        "error_files": error_files,
        "elapsed_min": elapsed_total / 60,
    }


In [ ]:

def print_summary(summaries: list[dict]) -> None:
    print("\n" + "=" * 60)
    print("ETL STAGE 1 SUMMARY")
    print("=" * 60)
    grand_total = 0
    for s in summaries:
        print(f"\nClass : {s['label'].upper()}")
        print(f"  Input videos   : {s['total_input']}")
        print(f"  Processed OK   : {s['ok']}")
        print(f"  Skipped (exist): {s['skipped']}")
        print(f"  Errors         : {s['errors']}")
        print(f"  Elapsed        : {s['elapsed_min']:.1f} min")
        grand_total += s["ok"] + s["skipped"]
        if s["error_files"]:
            print(f"  Error files:")
            for f in s["error_files"]:
                print(f"    - {f}")
    print(f"\nTotal videos in stage1/: {grand_total}")
    print("=" * 60)
    print("Stage 1 complete. Normalizacion ImageNet -> Stage 2.")


In [ ]:

def main():
    print("=" * 60)
    print("Aegis-Safe-Work | FallDetection ETL Stage 1")
    print("=" * 60)

    check_ffmpeg()

    summaries = []

    for label, sources in SOURCES.items():
        out_dir = OUTPUT_ROOT / label
        summary = process_class(label, sources, out_dir)
        summaries.append(summary)

    print_summary(summaries)


In [ ]:
main()

Aegis-Safe-Work | FallDetection ETL Stage 1
[OK] ffmpeg found.

[FALL] Collecting sources...
  Falls: 903 videos ['.avi', '.mp4']
[FALL] Total videos to process: 903
  [50/903] OK: Fall_0050.mp4 | rate=0.4 vid/s | ETA=35.3 min
  [100/903] OK: Fall_0100.mp4 | rate=0.5 vid/s | ETA=25.4 min
  [150/903] OK: Fall_0150.mp4 | rate=0.6 vid/s | ETA=19.9 min
  [200/903] OK: Fall_0200.mp4 | rate=0.7 vid/s | ETA=17.1 min
  [250/903] OK: Fall_0250.mp4 | rate=0.7 vid/s | ETA=14.8 min
  [300/903] OK: Fall_0300.mp4 | rate=0.8 vid/s | ETA=12.5 min
  [350/903] OK: Fall_0350.mp4 | rate=0.8 vid/s | ETA=11.4 min
  [400/903] TIMEOUT: C_M_223.mp4
  [450/903] OK: Fall_0450.mp4 | rate=0.7 vid/s | ETA=10.6 min
  [500/903] OK: Fall_0500.mp4 | rate=0.7 vid/s | ETA=9.5 min
  [550/903] OK: Fall_0550.mp4 | rate=0.7 vid/s | ETA=8.1 min
  [600/903] OK: Fall_0600.mp4 | rate=0.7 vid/s | ETA=6.7 min
  [650/903] OK: Fall_0650.mp4 | rate=0.8 vid/s | ETA=5.5 min
  [700/903] OK: Fall_0700.mp4 | rate=0.8 vid/s | ETA=4.4 min
 

In [ ]:
!ffprobe -v error -show_entries format=duration,size -of default=noprint_wrappers=1 \
    "/content/drive/MyDrive/Aegis_Safe_Work/raw/Falls/C_M_223.mp4"

duration=2.038000
size=145318947
